# Análisis Pokémon — Modelo No Relacional (MongoDB)

**Backend:** `pokemon_data.pokemon` · 1,350 documentos JSON · MongoDB

Este notebook analiza los datos Pokémon usando un modelo **documental**: cada Pokémon es un único objeto JSON con tipos, stats, habilidades, especie y movimientos **todo embebido** — sin necesidad de JOINs.

## ¿Cuándo usar datos no relacionales?
- Cuando necesitas recuperar una entidad completa de un golpe (APIs, dashboards)
- Búsquedas dentro de arrays embebidos con `$elemMatch`
- Datos con estructura variable o muy anidada

## Prerrequisitos
```bash
python fetch.py          # Descarga datos de PokéAPI (solo la primera vez)
python build_nosql.py    # Carga documentos en MongoDB
sudo systemctl start mongod  # MongoDB debe estar corriendo
```
> Consulta **MONGO_GUIDE.md** si es la primera vez que usas MongoDB.

---
**Secciones:** Conexión | Estructura de documento | Búsqueda por nombre | Legendarios | Aggregation Pipeline | Búsqueda por movimiento | Filtro por tipo

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib
import numpy as np

BG    = '#0f0f23'
PANEL = '#16213e'
FG    = '#e0e0e0'

matplotlib.rcParams.update({
    'figure.facecolor': BG,   'axes.facecolor':  PANEL,
    'axes.edgecolor':   '#444','axes.labelcolor': '#aaa',
    'xtick.color': FG,        'ytick.color':     FG,
    'text.color':  FG,        'grid.color':      '#2a2a4a',
    'grid.alpha':  0.7,
})


TYPE_COLORS = {
    'normal':'#A8A77A',   'fire':'#EE8130',    'water':'#6390F0',
    'electric':'#F7D02C', 'grass':'#7AC74C',   'ice':'#96D9D6',
    'fighting':'#C22E28', 'poison':'#A33EA1',  'ground':'#E2BF65',
    'flying':'#A98FF3',   'psychic':'#F95587', 'bug':'#A6B91A',
    'rock':'#B6A136',     'ghost':'#735797',   'dragon':'#6F35FC',
    'dark':'#705746',     'steel':'#B7B7CE',   'fairy':'#D685AD',
}

TYPE_ES = {
    'normal':'Normal',      'fire':'Fuego',     'water':'Agua',
    'electric':'Electrico', 'grass':'Planta',   'ice':'Hielo',
    'fighting':'Lucha',     'poison':'Veneno',  'ground':'Tierra',
    'flying':'Volador',     'psychic':'Psiquico','bug':'Bicho',
    'rock':'Roca',          'ghost':'Fantasma', 'dragon':'Dragon',
    'dark':'Siniestro',     'steel':'Acero',    'fairy':'Hada',
}

STAT_COLS   = ['hp', 'attack', 'defense', 'special-attack', 'special-defense', 'speed']
STAT_LABELS = ['HP', 'ATK', 'DEF', 'SP.ATK', 'SP.DEF', 'SPD']
GEN_MAP = {
    'generation-i':'Gen I',     'generation-ii':'Gen II',
    'generation-iii':'Gen III', 'generation-iv':'Gen IV',
    'generation-v':'Gen V',     'generation-vi':'Gen VI',
    'generation-vii':'Gen VII', 'generation-viii':'Gen VIII',
    'generation-ix':'Gen IX',
}
GEN_ORDER = ['Gen I','Gen II','Gen III','Gen IV','Gen V',
             'Gen VI','Gen VII','Gen VIII','Gen IX']

def bilabel(en):
    return TYPE_ES.get(en, en.title()) + ' / ' + en.title()

def badge_set(lbls, type_names, fontsize=9):
    for lbl, tipo in zip(lbls, type_names):
        c = TYPE_COLORS.get(tipo, '#777')
        lbl.set_bbox({'boxstyle':'round,pad=0.30', 'facecolor':c,
                      'edgecolor':'white', 'linewidth':0.5})
        lbl.set_color('white')
        lbl.set_fontsize(fontsize)
        lbl.set_fontweight('bold')

def clean_ax(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#333')
    ax.spines['bottom'].set_color('#333')

def minor_grid(ax, nx, ny):
    ax.set_xticks(np.arange(-0.5, nx, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, ny, 1), minor=True)
    ax.grid(which='minor', color='#333', linewidth=0.6)
    ax.tick_params(which='minor', bottom=False, left=False)

print('Setup listo.')


## 1. Conexión a MongoDB

Conectamos a la base de datos `pokemon_data`, colección `pokemon`.
Cada documento equivale a una fila de la tabla `pokemon` más todos sus JOINs ya resueltos.

La URL de conexión se puede cambiar con la variable de entorno `MONGO_URI`.

In [ ]:
import os
from pymongo import MongoClient

# Conexión — lee MONGO_URI del entorno o usa localhost por defecto
_client = MongoClient(os.getenv("MONGO_URI", "mongodb://localhost:27017"))
col = _client["pokemon_data"]["pokemon"]

total = col.count_documents({})
print(f"Conexión OK  —  {total} documentos en pokemon_data.pokemon")
print("Consultas disponibles en las celdas siguientes.")

## 2. Estructura de un Documento

Antes de consultar, veamos cómo está organizado un documento completo.
Todos los campos disponibles para filtrar o proyectar están aquí.

In [ ]:
import json

# Ver todos los campos de un documento (sin moves para no saturar la pantalla)
doc_ejemplo = col.find_one({'name': 'charizard'}, {'_id': 0, 'moves': 0})
print(json.dumps(doc_ejemplo, indent=2, ensure_ascii=False))

In [ ]:
# Campos disponibles en el documento completo (incluyendo moves)
doc_full = col.find_one({'name': 'charizard'})
print('Campos de primer nivel:', list(doc_full.keys()))
print('Campos en types[0]:    ', list(doc_full['types'][0].keys()))
print('Campos en abilities[0]:', list(doc_full['abilities'][0].keys()))
print('Campos en species:     ', list(doc_full['species'].keys()))
print('Campos en moves[0]:    ', list(doc_full['moves'][0].keys()))
print(f'\nTotal movimientos embebidos: {len(doc_full["moves"])}')

## 3. Búsqueda por Nombre o ID

In [ ]:
# Búsqueda por nombre o ID — todos los datos en un solo documento
# Equivalente a: SELECT * FROM pokemon JOIN pokemon_stats JOIN pokemon_types JOIN species WHERE name='charizard'
doc = col.find_one({"name": "charizard"}, {"_id": 0, "moves": 0})
if doc:
    print(f"#{doc['id']} {doc['name'].title()}")
    print(f"Tipos:    {' / '.join(t['name'] for t in doc['types'])}")
    print(f"Stats:    {doc['stats']}")
    print(f"BST:      {sum(doc['stats'].values())}")
    print(f"Especie:  gen={doc['species'].get('generation')} | legendario={doc['species'].get('is_legendary')}")
    print(f"Flavor:   {doc['species'].get('flavor_text', '')[:120]}…")
else:
    print("No encontrado")

## 4. Legendarios y Míticos

In [ ]:
# Legendarios y míticos con sus tipos
# Equivalente a: SELECT name FROM pokemon JOIN species WHERE is_legendary=1
# La ventaja: todo está en un solo documento, sin JOIN
legendarios = list(col.find(
    {"species.is_legendary": True},
    {"_id": 0, "id": 1, "name": 1, "types": 1, "stats": 1, "species.generation": 1}
))
df_leg = pd.DataFrame([
    {
        "id":         r["id"],
        "name":       r["name"],
        "tipos":      " / ".join(t["name"] for t in r.get("types", [])),
        "bst":        sum(r.get("stats", {}).values()),
        "generacion": r.get("species", {}).get("generation", "?"),
    }
    for r in legendarios
])
print(f"Total legendarios: {len(df_leg)}")
df_leg.sort_values("bst", ascending=False).reset_index(drop=True).head(15)

## 5. Aggregation Pipeline — BST por Tipo

In [ ]:
# Aggregation Pipeline: BST promedio por tipo primario
# Equivalente al heatmap de stats (sección 4), pero calculado en MongoDB
# $objectToArray convierte {"hp":45, "attack":49,...} en [{k:"hp",v:45}, ...]
pipeline = [
    {"$addFields": {
        "primary_type": {"$arrayElemAt": ["$types", 0]},
        "bst": {"$sum": {
            "$map": {
                "input": {"$objectToArray": "$stats"},
                "as": "s",
                "in": "$$s.v"
            }
        }}
    }},
    {"$group": {
        "_id": "$primary_type.name",
        "avg_bst":  {"$avg": "$bst"},
        "max_bst":  {"$max": "$bst"},
        "count":    {"$sum": 1}
    }},
    {"$sort": {"avg_bst": -1}}
]
df_agg = pd.DataFrame(list(col.aggregate(pipeline)))
df_agg.columns = ["tipo", "bst_promedio", "bst_maximo", "n_pokemon"]
df_agg["bst_promedio"] = df_agg["bst_promedio"].round(1)
df_agg.reset_index(drop=True)

## 6. Búsqueda por Movimiento (`$elemMatch`)

In [ ]:
# Pokémon que aprenden un movimiento específico
# En SQL esto requiere: JOIN pokemon_moves pm ON p.id=pm.pokemon_id WHERE pm.move_name=...
# En MongoDB basta un $elemMatch sobre el array embebido "moves"
move = "earthquake"
result = list(col.find(
    {"moves": {"$elemMatch": {"name": move}}},
    {"_id": 0, "name": 1, "types": 1}
))
print(f"{len(result)} Pokémon aprenden '{move}'")
df_move = pd.DataFrame([
    {"name": r["name"], "tipos": " / ".join(t["name"] for t in r.get("types", []))}
    for r in result
])
df_move.sort_values("name").reset_index(drop=True).head(20)

## 7. Filtro Combinado — Tipo y BST

Ejemplo de consulta con múltiples condiciones: tipo de Pokémon y umbral de BST mínimo.
En SQL requeriría un JOIN + subconsulta; aquí es un solo `find()`.

In [ ]:
# Pokémon de tipo dragón con BST > 500
# En SQL: SELECT ... FROM pokemon JOIN pokemon_types JOIN pokemon_stats
#          WHERE type_name='dragon' GROUP BY id HAVING SUM(base_value)>500
tipo_filtro = 'dragon'
bst_minimo  = 500

resultado = list(col.find(
    {'types.name': tipo_filtro},
    {'_id': 0, 'name': 1, 'types': 1, 'stats': 1, 'species.generation': 1}
))
df_filtro = pd.DataFrame([
    {
        'name':       r['name'],
        'tipos':      ' / '.join(t['name'] for t in r['types']),
        'bst':        sum(r['stats'].values()),
        'generacion': r.get('species', {}).get('generation', '?'),
    }
    for r in resultado
])
df_filtro = df_filtro[df_filtro['bst'] > bst_minimo].sort_values('bst', ascending=False)
print(f'{len(df_filtro)} Pokémon de tipo {tipo_filtro} con BST > {bst_minimo}')
df_filtro.reset_index(drop=True)